# LiTS Tumor Detection with MedSAM
## A notebook for visualizing NII files and detecting tumors using MedSAM

This notebook demonstrates:
- Loading and visualizing .nii files from the LiTS dataset
- Using MedSAM (Medical Segment Anything Model) for tumor detection
- Visualizing detection results with overlays

## Section 1: Setup and Installation

In [1]:
# Install required packages
!pip install nibabel matplotlib numpy torch torchvision
!pip install segment-anything
!pip install opencv-python-headless

   ---------------------------------------- 0.0/38.9 MB ? eta -:--:--
   - -------------------------------------- 1.3/38.9 MB 9.6 MB/s eta 0:00:04
   -- ------------------------------------- 2.9/38.9 MB 9.3 MB/s eta 0:00:04
   ---- ----------------------------------- 4.7/38.9 MB 8.1 MB/s eta 0:00:05
   ------ --------------------------------- 6.6/38.9 MB 8.4 MB/s eta 0:00:04
   -------- ------------------------------- 7.9/38.9 MB 7.6 MB/s eta 0:00:05
   --------- ------------------------------ 9.4/38.9 MB 7.7 MB/s eta 0:00:04
   ----------- ---------------------------- 11.3/38.9 MB 7.7 MB/s eta 0:00:04
   ------------- -------------------------- 12.8/38.9 MB 7.8 MB/s eta 0:00:04
   --------------- ------------------------ 14.7/38.9 MB 7.8 MB/s eta 0:00:04
   ---------------- ----------------------- 16.0/38.9 MB 7.7 MB/s eta 0:00:03
   ------------------ --------------------- 17.6/38.9 MB 7.6 MB/s eta 0:00:03
   ------------------- -------------------- 18.9/38.9 MB 7.5 MB/s eta 0:00:03


## Section 2: Import Libraries

In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib
from google.colab import files
import torch
import cv2
from segment_anything import sam_model_registry, SamPredictor
import urllib.request
from pathlib import Path

print("All libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

ModuleNotFoundError: No module named 'google.colab'

## Section 3: Download and Setup MedSAM

In [3]:
# Download MedSAM checkpoint
medsam_checkpoint_path = "medsam_vit_b.pth"

print("Downloading MedSAM checkpoint...")
if not os.path.exists(medsam_checkpoint_path):
    # Using gdown for Google Drive downloads
    !pip install gdown
    !gdown 1UAmWL88roYR7wKlnApw5Bcuzf2iQgk6_
    print("MedSAM checkpoint downloaded!")
else:
    print("MedSAM checkpoint already exists!")

MedSAM checkpoint downloaded!


Downloading...
From (original): https://drive.google.com/uc?id=1UAmWL88roYR7wKlnApw5Bcuzf2iQgk6_
From (redirected): https://drive.google.com/uc?id=1UAmWL88roYR7wKlnApw5Bcuzf2iQgk6_&confirm=t&uuid=c3162b83-d1f6-4225-94cc-d66967b0aef4
To: c:\Users\HP\Documents\medsam_vit_b.pth

  0%|          | 0.00/375M [00:00<?, ?B/s]
  0%|          | 524k/375M [00:00<10:10, 613kB/s]
  0%|          | 1.05M/375M [00:01<05:46, 1.08MB/s]
  0%|          | 1.57M/375M [00:01<04:06, 1.52MB/s]
  1%|          | 2.10M/375M [00:01<03:00, 2.06MB/s]
  1%|          | 3.67M/375M [00:01<01:30, 4.11MB/s]
  1%|▏         | 4.72M/375M [00:01<01:14, 4.99MB/s]
  2%|▏         | 5.77M/375M [00:01<01:13, 5.01MB/s]
  2%|▏         | 7.34M/375M [00:02<00:58, 6.28MB/s]
  2%|▏         | 8.39M/375M [00:02<00:54, 6.77MB/s]
  3%|▎         | 9.44M/375M [00:02<00:50, 7.26MB/s]
  3%|▎         | 10.5M/375M [00:02<00:48, 7.57MB/s]
  3%|▎         | 11.5M/375M [00:02<00:45, 8.03MB/s]
  3%|▎         | 12.6M/375M [00:02<00:43, 8.28MB/s]
  4%|▎

## Section 4: Upload or Specify NII File Path

In [4]:
# OPTION 1: Upload file directly (Uncomment to use)
print("Please upload a .nii or .nii.gz file from the LiTS dataset")
uploaded = files.upload()
nii_file = list(uploaded.keys())[0]

# OPTION 2: Mount Google Drive (Uncomment to use)
# from google.colab import drive
# drive.mount('/content/drive')
# nii_file = '/content/drive/MyDrive/path/to/your/file.nii'

print(f"Using file: {nii_file}")

Please upload a .nii or .nii.gz file from the LiTS dataset


NameError: name 'files' is not defined

## Section 5: Load and Visualize NII File

In [ ]:
def load_nii_file(file_path):
    """Load a NII file and return the data array"""
    try:
        nii_img = nib.load(file_path)
        nii_data = nii_img.get_fdata()
        print(f"NII file loaded successfully!")
        print(f"Shape: {nii_data.shape}")
        print(f"Data type: {nii_data.dtype}")
        print(f"Value range: [{nii_data.min()}, {nii_data.max()}]")
        return nii_data
    except Exception as e:
        print(f"Error loading NII file: {e}")
        return None

# Load the NII file
nii_data = load_nii_file(nii_file)

In [ ]:
def visualize_nii_slices(nii_data, num_slices=6):
    """Visualize multiple slices from the NII volume"""
    if nii_data is None:
        print("No data to visualize")
        return
    
    # Get slices at regular intervals
    depth = nii_data.shape[2]
    slice_indices = np.linspace(0, depth-1, num_slices, dtype=int)
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    for idx, slice_idx in enumerate(slice_indices):
        slice_data = nii_data[:, :, slice_idx]
        axes[idx].imshow(slice_data.T, cmap='gray', origin='lower')
        axes[idx].set_title(f'Slice {slice_idx}/{depth}')
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()

# Visualize slices
if nii_data is not None:
    visualize_nii_slices(nii_data)

## Section 6: Initialize MedSAM Model

In [ ]:
def initialize_medsam(checkpoint_path):
    """Initialize MedSAM model"""
    try:
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Using device: {device}")
        
        # MedSAM uses SAM ViT-B architecture
        sam_model = sam_model_registry["vit_b"](checkpoint=checkpoint_path)
        sam_model.to(device)
        sam_model.eval()
        
        predictor = SamPredictor(sam_model)
        print("MedSAM model initialized successfully!")
        return predictor, device
    except Exception as e:
        print(f"Error initializing MedSAM: {e}")
        return None, None

# Initialize model
predictor, device = initialize_medsam(medsam_checkpoint_path)

## Section 7: Preprocessing Functions

In [ ]:
def preprocess_slice_for_medsam(slice_data):
    """Preprocess a CT slice for MedSAM"""
    # Normalize to 0-255
    slice_normalized = (slice_data - slice_data.min()) / (slice_data.max() - slice_data.min())
    slice_8bit = (slice_normalized * 255).astype(np.uint8)
    
    # Convert to RGB (MedSAM expects 3 channels)
    slice_rgb = cv2.cvtColor(slice_8bit, cv2.COLOR_GRAY2RGB)
    
    return slice_rgb

def get_bounding_box_prompt(image_shape, center_ratio=0.5, size_ratio=0.6):
    """Generate a bounding box prompt for the center of the image"""
    h, w = image_shape[:2]
    box_w = int(w * size_ratio)
    box_h = int(h * size_ratio)
    
    x1 = int(w * center_ratio - box_w // 2)
    y1 = int(h * center_ratio - box_h // 2)
    x2 = x1 + box_w
    y2 = y1 + box_h
    
    return np.array([x1, y1, x2, y2])

print("Preprocessing functions defined!")

## Section 8: Tumor Detection Functions

In [ ]:
def detect_tumor_in_slice(predictor, slice_data, device):
    """Detect tumor in a single slice using MedSAM"""
    if predictor is None:
        print("Predictor not initialized!")
        return None, None
    
    # Preprocess
    slice_rgb = preprocess_slice_for_medsam(slice_data)
    
    # Set image for predictor
    predictor.set_image(slice_rgb)
    
    # Generate bounding box prompt (targeting center of image)
    box_prompt = get_bounding_box_prompt(slice_rgb.shape)
    
    # Predict
    with torch.no_grad():
        masks, scores, logits = predictor.predict(
            box=box_prompt,
            multimask_output=True
        )
    
    # Get best mask
    best_mask_idx = np.argmax(scores)
    best_mask = masks[best_mask_idx]
    best_score = scores[best_mask_idx]
    
    return best_mask, best_score

def visualize_detection(slice_data, mask, score, slice_idx):
    """Visualize the original slice and detected tumor mask"""
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Original slice
    axes[0].imshow(slice_data.T, cmap='gray', origin='lower')
    axes[0].set_title(f'Original Slice {slice_idx}')
    axes[0].axis('off')
    
    # Mask
    axes[1].imshow(mask.T, cmap='jet', origin='lower')
    axes[1].set_title(f'Detected Mask (Score: {score:.3f})')
    axes[1].axis('off')
    
    # Overlay
    axes[2].imshow(slice_data.T, cmap='gray', origin='lower')
    axes[2].imshow(mask.T, cmap='jet', alpha=0.4, origin='lower')
    axes[2].set_title('Overlay')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()

print("Detection functions defined!")

## Section 9: Run Detection on Sample Slices

In [ ]:
def process_volume(nii_data, predictor, device, num_slices=3):
    """Process multiple slices from the volume"""
    if nii_data is None or predictor is None:
        print("Data or predictor not available!")
        return
    
    depth = nii_data.shape[2]
    # Select slices with potential tumors (middle region)
    slice_indices = np.linspace(depth//3, 2*depth//3, num_slices, dtype=int)
    
    print(f"\nProcessing {num_slices} slices for tumor detection...")
    
    for slice_idx in slice_indices:
        print(f"\n{'='*60}")
        print(f"Processing slice {slice_idx}/{depth}")
        print('='*60)
        
        slice_data = nii_data[:, :, slice_idx]
        mask, score = detect_tumor_in_slice(predictor, slice_data, device)
        
        if mask is not None:
            visualize_detection(slice_data, mask, score, slice_idx)
            print(f"Detection confidence: {score:.3f}")
            print(f"Tumor area: {mask.sum()} pixels")

# Run detection
if nii_data is not None and predictor is not None:
    print("\n" + "="*60)
    print("STARTING TUMOR DETECTION")
    print("="*60)
    process_volume(nii_data, predictor, device, num_slices=3)
else:
    print("Please ensure NII data is loaded and model is initialized!")

## Instructions for Use

1. **Upload Dataset**: Run Section 4 to upload your LiTS NII file
2. **Load & Visualize**: Section 5 loads and shows slices from your volume
3. **Initialize Model**: Section 6 loads the MedSAM model
4. **Run Detection**: Section 9 detects tumors in selected slices

### Customization:
- Adjust `num_slices` parameter to process more/fewer slices
- Modify `slice_indices` to target specific regions
- Change bounding box parameters for different detection areas

### Notes:
- GPU recommended for faster processing
- Results vary based on image quality and tumor characteristics
- MedSAM works best with clear tumor regions